# CCC–Phenotype V0：可逐格运行流程

唯一主线：**当前研究 ST（RCTD + COMMOT）→ BayesPrism bulk → 患者级 exact directed CCC → 图正则稀疏 Cox**。

不可更改的规则：研究哪套 ST，哪套 ST 就是该次分析唯一的空间通讯锚点；不存在 ref ST 与 validation ST 的区分。患者 CCC 为

$$C_{s,a,b,p}=W^{ST}_{a,b,p}\frac{M^{RNA}_{s,a,b,p}+\epsilon}{M^{ST}_{a,b,p}+\epsilon}.$$

第一次使用时只需要集中修改下面的参数单元格。耗时步骤都有缓存；`OVERWRITE=False` 时不会覆盖已有结果。

In [ ]:
# ======================== 唯一需要集中修改的参数单元格 ========================
from pathlib import Path

PROJECT_ROOT = Path('/home/xueshuailin/CCC_Phe')
ST_ROOT = Path('/home/nas3/biod/xueshuailin/New_Projects/data/SpaPheno_liver_cancer_ST/samples')
ST_SAMPLES = ['HCC-1L', 'HCC-2L', 'HCC-3L', 'HCC-4L', 'cHC-1L', 'ICC-1L']
TCGA_RAW_COUNTS = Path('/home/nas3/biod/xueshuailin/New_Projects/data/TCGA_LIHC/raw_star_counts')
TCGA_FILE_MAP = Path('/home/nas3/biod/xueshuailin/New_Projects/data/TCGA_LIHC/source_metadata/gdc_rnaseq_file_sample_map.tsv')
CLINICAL = Path('/home/nas3/biod/xueshuailin/New_Projects/data/TCGA_LIHC/clinical_aligned_plus.tsv')

# 必填：带 raw counts、细胞类型注释的肝癌 scRNA-seq h5ad；同时供 RCTD 和 BayesPrism 使用。
SC_REFERENCE_H5AD = Path(
    '/home/nas3/biod/xueshuailin/New_Projects/data/GSE189903_scRNA/processed/'
    'GSE189903_liver_reference_raw_counts.h5ad'
)  # 例如 Path('/path/to/liver_scRNA_reference.h5ad')
SC_COUNTS_LAYER = None  # 若 raw counts 在 .X，填 None
CELL_TYPE_KEY = 'cell_type'
CELL_STATE_KEY = None       # 没有更细 state 时保持 None
MALIGNANT_LABEL = 'Cancer epithelial'  # 必须与 CELL_TYPE_KEY 中的肿瘤细胞标签完全一致

WORK_ROOT = Path('/home/nas3/biod/xueshuailin/CCC_Phe/data/processed/lihc_v0')
RESULT_ROOT = Path('/home/nas3/biod/xueshuailin/CCC_Phe/results/lihc_v0')
OVERWRITE = False
RANDOM_SEED = 20260730

# RCTD / COMMOT
RCTD_CORES = 8
COMMOT_CHUNK_SIZE = 100
COMMOT_DISTANCE_THRESHOLD = None  # None: 每张切片按最近邻距离自动估计
MIN_ST_SAMPLES_PER_CCC = 4
COMMOT_COT_EPS_P = 0.1
COMMOT_COT_RHO = 10.0
COMMOT_COT_NITERMAX = 10000

# BayesPrism
BAYESPRISM_CORES = 8
BAYESPRISM_CHAIN_LENGTH = 1000
BAYESPRISM_BURN_IN = 500
MIN_CELL_FRACTION = 0.001
EPSILON = 1e-8

# Cox
ENDPOINT = 'OS'
COX_DEVICE = 'cuda:1'  # 无 GPU 可改为 'cpu'
OUTER_FOLDS = 5
INNER_FOLDS = 4
LAMBDA_FRACTIONS = [0.60, 0.40, 0.30, 0.20, 0.10]
GRAPH_LAMBDAS = [0.00, 0.01, 0.03, 0.05]
TUNING_CINDEX_TOLERANCE = 0.01  # 距最佳 C-index 不超过此值时优先选更稀疏模型
STABILITY_REFINEMENT_FRACTIONS = [0.10, 0.075, 0.05]
STABILITY_REFINEMENT_GRAPH = 0.05
REFERENCE_MEAN_CINDEX = 0.618  # 原 0.20/0.05 模型的 held-out mean C-index
TARGET_STABLE_CCC_RANGE = (30, 100)
RIDGE_LAMBDA = 1e-5
STABILITY_REPLICATES = 50
STABILITY_THRESHOLD = 0.70
# ============================================================================


In [ ]:
# 0. 导入项目函数并定义所有检查点路径
import json, subprocess, sys
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from cccphe.config import load_config
from cccphe.workflow import (
    prepare_single_cell_reference, prepare_visium_rctd_inputs, run_rctd,
    build_commot_anchor, prepare_tcga_bulk_counts, run_bayesprism,
    build_patient_tensor,
)
from cccphe.network_cox import run_network_sparse_cox

REFERENCE_DIR = WORK_ROOT / 'single_cell_reference'
VISIUM_INPUT_DIR = WORK_ROOT / 'rctd_inputs' / 'spatial'
RCTD_DIR = WORK_ROOT / 'rctd'
COMMOT_DIR = WORK_ROOT / 'commot_anchor'
BULK_COUNT_DIR = WORK_ROOT / 'bayesprism' / 'bulk'
BAYESPRISM_DIR = WORK_ROOT / 'bayesprism' / 'output'
TENSOR_DIR = WORK_ROOT / 'communication_tensor'
print('WORK_ROOT =', WORK_ROOT)


In [ ]:
# 1. 输入与软件审计（轻量；建议每次先运行）
required_paths = {
    'ST_ROOT': ST_ROOT, 'TCGA_RAW_COUNTS': TCGA_RAW_COUNTS,
    'TCGA_FILE_MAP': TCGA_FILE_MAP, 'CLINICAL': CLINICAL,
}
for sample in ST_SAMPLES:
    required_paths[f'ST:{sample}'] = ST_ROOT / sample / 'filtered_feature_bc_matrix.h5'
audit = pd.DataFrame([(name, str(path), path.exists()) for name, path in required_paths.items()],
                     columns=['input', 'path', 'exists'])
display(audit)
assert audit.exists.all(), '有输入文件缺失，请先修正参数单元格。'
assert SC_REFERENCE_H5AD is not None, '请在参数单元格填写 SC_REFERENCE_H5AD。'
assert Path(SC_REFERENCE_H5AD).exists(), f'单细胞参考不存在：{SC_REFERENCE_H5AD}'

for package in ['spacexr', 'BayesPrism']:
    check = subprocess.run(['Rscript', '-e', f'quit(status=!requireNamespace(\"{package}\", quietly=TRUE))'])
    assert check.returncode == 0, f'R 包 {package} 尚未安装。'
print('输入与 R 依赖检查通过。')


## A. 当前 ST 建立唯一通讯锚点
RCTD 只负责给每个 Visium spot 提供细胞类型概率；真正的 `sender → receiver → LR` 方向由这六张当前 ST 上的 COMMOT 给出。

In [ ]:
# 2. 一次性准备单细胞参考和六张 Visium 的 RCTD 输入
prepare_single_cell_reference(
    Path(SC_REFERENCE_H5AD), REFERENCE_DIR, cell_type_key=CELL_TYPE_KEY,
    cell_state_key=CELL_STATE_KEY, counts_layer=SC_COUNTS_LAYER, overwrite=OVERWRITE,
)
prepare_visium_rctd_inputs(ST_ROOT, ST_SAMPLES, VISIUM_INPUT_DIR, overwrite=OVERWRITE)
print('RCTD 输入准备完成。')


In [ ]:
# 3. 逐张切片运行 RCTD（耗时；已有 weights.tsv.gz 时可跳过）
missing = [s for s in ST_SAMPLES if not (RCTD_DIR / s / 'weights.tsv.gz').exists()]
if missing:
    run_rctd(PROJECT_ROOT, REFERENCE_DIR, VISIUM_INPUT_DIR, RCTD_DIR, missing, cores=RCTD_CORES)
else:
    print('六张切片的 RCTD 结果均已存在，跳过。')
for sample in ST_SAMPLES:
    w = pd.read_csv(RCTD_DIR / sample / 'weights.tsv.gz', sep='\t')
    assert w.shape[0] > 0 and w.iloc[:, 1:].notna().all().all()
print('RCTD 输出检查通过。')


In [ ]:
# 4. 在同一批 ST 上运行 COMMOT，并跨切片取中位数得到 W_ST 与 M_ST（最耗时）
anchor_path = build_commot_anchor(
    VISIUM_INPUT_DIR, RCTD_DIR, ST_SAMPLES, COMMOT_DIR,
    minimum_st_samples=MIN_ST_SAMPLES_PER_CCC, chunk_size=COMMOT_CHUNK_SIZE,
    distance_threshold=COMMOT_DISTANCE_THRESHOLD, cot_eps_p=COMMOT_COT_EPS_P,
    cot_rho=COMMOT_COT_RHO, cot_nitermax=COMMOT_COT_NITERMAX, overwrite=OVERWRITE,
)
anchor = pd.read_parquet(anchor_path)
display(anchor.groupby('confident').size().rename('CCC_rows'))
display(anchor[anchor.confident].head())
assert anchor.confident.any(), '没有通过跨切片置信度要求的 ST CCC。'


## B. TCGA-LIHC bulk 构建患者级 CCC
BayesPrism 恢复患者细胞类型特异表达；它不会重新决定通讯方向。患者端只通过相对分子支持对当前 ST 的 `W_ST` 进行重标定。

In [ ]:
# 5. 从 GDC STAR raw counts 准备 BayesPrism mixture（只保留 Primary Tumor，并按患者 ID 整理）
prepare_tcga_bulk_counts(
    TCGA_RAW_COUNTS, TCGA_FILE_MAP, REFERENCE_DIR, BULK_COUNT_DIR, overwrite=OVERWRITE,
)
bulk_samples = pd.read_csv(BULK_COUNT_DIR / 'samples.tsv', header=None)[0]
clinical_samples = set(pd.read_csv(CLINICAL, sep='\t').Sample_ID.astype(str))
print('BayesPrism bulk patients:', len(bulk_samples))
print('Matched clinical patients:', bulk_samples.astype(str).isin(clinical_samples).sum())
assert bulk_samples.astype(str).isin(clinical_samples).all()


In [ ]:
# 6. 运行 BayesPrism（耗时；已有 fractions_final.tsv.gz 时可跳过）
if not (BAYESPRISM_DIR / 'fractions_final.tsv.gz').exists() or OVERWRITE:
    run_bayesprism(
        PROJECT_ROOT, REFERENCE_DIR, BULK_COUNT_DIR, BAYESPRISM_DIR, COMMOT_DIR / 'lr_genes.txt',
        malignant_label=MALIGNANT_LABEL, cores=BAYESPRISM_CORES,
        chain_length=BAYESPRISM_CHAIN_LENGTH, burn_in=BAYESPRISM_BURN_IN, seed=RANDOM_SEED,
    )
else:
    print('BayesPrism 输出已存在，跳过。')
fractions = pd.read_csv(BAYESPRISM_DIR / 'fractions_final.tsv.gz', sep='\t')
display(fractions.head())
assert fractions.Sample_ID.is_unique


In [ ]:
# 7. 构建并保存患者 exact CCC 张量
# C = W_ST * (M_RNA + epsilon) / (M_ST + epsilon)
tensor_dir = build_patient_tensor(
    anchor_path, BAYESPRISM_DIR, CLINICAL, TENSOR_DIR,
    minimum_cell_fraction=MIN_CELL_FRACTION, epsilon=EPSILON, overwrite=OVERWRITE,
)
audit = json.loads((tensor_dir / 'audit.json').read_text())
display(pd.Series(audit, name='value').to_frame())
tensor_samples = pd.read_csv(tensor_dir / 'samples.tsv', sep='\t').Sample_ID
clinical = pd.read_csv(CLINICAL, sep='\t').set_index('Sample_ID')
assert tensor_samples.is_unique and clinical.reindex(tensor_samples).index.notna().all()
assert audit['st_rule'] == 'current study ST is the only spatial anchor'


## C. 直接寻找表型相关 exact CCC
每个 `(sender → receiver, LR)` 是一个 Cox 特征。不做 CP/NMF，不先划 PR-high/low 区域。先运行 smoke test，确认输入与模型连通，再单独运行完整模型。

In [ ]:
# 8. 将参数写入内存配置，并运行小规模 smoke test
from copy import deepcopy
cfg = load_config(PROJECT_ROOT / 'config' / 'config.yaml')
cfg['project']['random_seed'] = RANDOM_SEED
cfg['analysis_paths']['communication_tensor'] = str(TENSOR_DIR)
cfg['analysis_paths']['clinical'] = str(CLINICAL)
cfg['paths']['results'] = str(RESULT_ROOT)
cfg['phenotype']['primary_endpoint'] = ENDPOINT
cfg['network_cox'].update({
    'device': COX_DEVICE, 'outer_folds': OUTER_FOLDS, 'inner_folds': INNER_FOLDS,
    'lambda_fractions': LAMBDA_FRACTIONS, 'graph_lambdas': GRAPH_LAMBDAS,
    'tuning_cindex_tolerance': TUNING_CINDEX_TOLERANCE,
    'ridge_lambda': RIDGE_LAMBDA, 'stability_replicates': STABILITY_REPLICATES,
    'stability_threshold': STABILITY_THRESHOLD,
})
smoke_cfg = deepcopy(cfg)
smoke_cfg['paths']['results'] = str(RESULT_ROOT / 'smoke_test')
smoke_output = run_network_sparse_cox(smoke_cfg, smoke=True)
print('Smoke test complete:', smoke_output)


In [ ]:
# 9. 先比较完整 lambda × graph 网格，再以稳定 CCC 数量做小范围 refinement

grid_cfg = deepcopy(cfg)
grid_cfg['paths']['results'] = str(RESULT_ROOT / 'hyperparameter_search')
grid_output = run_network_sparse_cox(grid_cfg, smoke=False)
grid = pd.read_csv(grid_output / 'hyperparameter_grid_summary.tsv', sep='\t')
display(grid.sort_values(['mean_validation_cindex', 'median_nonzero'], ascending=[False, True]))

refinement = []
for fraction in STABILITY_REFINEMENT_FRACTIONS:
    trial_cfg = deepcopy(cfg)
    trial_cfg['paths']['results'] = str(RESULT_ROOT / 'stability_refinement' / f'lambda_{fraction:g}')
    trial_cfg['network_cox']['lambda_fractions'] = [fraction]
    trial_cfg['network_cox']['graph_lambdas'] = [STABILITY_REFINEMENT_GRAPH]
    trial_output = run_network_sparse_cox(trial_cfg, smoke=False)
    trial_summary = json.loads((trial_output / 'summary.json').read_text())
    trial_all = pd.read_csv(trial_output / 'all_ccc_coefficients.tsv.gz', sep='\t')
    refinement.append({
        'lambda_fraction': fraction, 'graph_lambda': STABILITY_REFINEMENT_GRAPH,
        'mean_fold_cindex': trial_summary['mean_fold_cindex'],
        'full_model_selected_ccc': trial_summary['full_model_selected_ccc'],
        'stable_ge_0.70': int((trial_all.sign_selection_probability >= 0.70).sum()),
        'stable_ge_0.80': int((trial_all.sign_selection_probability >= 0.80).sum()),
        'output': str(trial_output),
    })
refinement = pd.DataFrame(refinement)
refinement.to_csv(RESULT_ROOT / 'cox_stability_refinement.tsv', sep='\t', index=False)
low, high = TARGET_STABLE_CCC_RANGE
acceptable = refinement[refinement.mean_fold_cindex >= REFERENCE_MEAN_CINDEX]
if acceptable.empty:
    raise RuntimeError('没有 refinement 模型保持基准 C-index；不要为稳定性牺牲泛化性能。')
in_target = acceptable[acceptable['stable_ge_0.70'].between(low, high)]
pool = in_target if not in_target.empty else acceptable.assign(
    target_distance=lambda x: (x['stable_ge_0.70'] - x['stable_ge_0.70'].clip(low, high)).abs()
)
sort_columns = ['stable_ge_0.80', 'mean_fold_cindex'] if not in_target.empty else ['target_distance', 'stable_ge_0.80', 'mean_fold_cindex']
ascending = [False, False] if not in_target.empty else [True, False, False]
selected_trial = pool.sort_values(sort_columns, ascending=ascending).iloc[0]
full_output = Path(selected_trial.output)
(RESULT_ROOT / 'selected_cox_model.json').write_text(json.dumps({
    'lambda_fraction': float(selected_trial.lambda_fraction),
    'graph_lambda': float(selected_trial.graph_lambda),
    'mean_fold_cindex': float(selected_trial.mean_fold_cindex),
    'stable_ge_0.70': int(selected_trial['stable_ge_0.70']),
    'stable_ge_0.80': int(selected_trial['stable_ge_0.80']),
    'output': str(full_output),
}, indent=2) + '\n')
summary = json.loads((full_output / 'summary.json').read_text())
display(refinement)
print('Selected stability-guided model:', full_output)
display(pd.Series(summary, name='value').to_frame())


In [ ]:
# 10. 查看最终风险/保护 CCC
stable = pd.read_csv(full_output / 'stable_ccc.tsv', sep='\t')
show = ['sender', 'receiver', 'ligand', 'receptor', 'pathway', 'direction',
        'full_coefficient', 'sign_selection_probability', 'stability_importance']
display(stable[show].sort_values('stability_importance', ascending=False).head(50))
print('全部 CCC 系数：', full_output / 'all_ccc_coefficients.tsv.gz')
print('稳定 CCC：', full_output / 'stable_ccc.tsv')
print('交叉验证：', full_output / 'fold_metrics.tsv')


## D. ST functional niche proof-of-concept（bulk/Cox 完成后的独立扩展）

这里只使用 `stable_ccc.tsv` 中稳定性 ≥ 0.70 且 Cox 系数 < 0 的保护性 exact CCC。先用 COMMOT 定向流、RCTD sender/receiver 概率、NMF 和空间 hotspot 完成**无标签发现**；TLS 标注不会参与 CCC 筛选、NMF 或 K 的选择，只能在 discovery manifest 写出后用于验证。未进入局部连续 hotspot 的 spot 保持 background，不强制分配 niche。


In [ ]:
# 11. ST niche 参数（只需在这里调整这一扩展的参数）
selected_cox = json.loads((RESULT_ROOT / 'selected_cox_model.json').read_text())
STABLE_CCC_PATH = Path(selected_cox['output']) / 'stable_ccc.tsv'
NICHE_OUTPUT = RESULT_ROOT / 'st_functional_niche'
PROTECTIVE_STABILITY = 0.70
NMF_K_VALUES = range(2, 9)
NMF_RESTARTS = 20
NMF_MAX_ITER = 2000
NMF_SELECTED_K = None       # None：只按重复初始化稳定性 + 重构误差自动选择
HOTSPOT_QUANTILE = 0.90
MIN_NICHE_SPOTS = 10
NICHE_OVERWRITE = False

# 后置验证文件必须有 sample_id、spot_id、tls 三列；没有逐 spot 官方标注时保持 None。
# 绝不能用 TLS gene score 在 discovery 前制造或筛选 niche。
TLS_ANNOTATION = ST_ROOT.parent / 'annotations' / 'tls_spot_annotations.tsv'  # 官方 demo：cHC-1L
TLS_PERMUTATIONS = 1000

from cccphe.st_niche import (
    load_protective_cccs, run_st_niche_discovery, validate_tls_niches,
    plot_nmf_diagnostics, plot_program_loadings, plot_spatial_programs, plot_tls_overlap,
)


In [ ]:
# 12. 只按 Cox 结果选择稳定保护性 CCC（不看 TLS，不预筛细胞类型）
protective_cccs = load_protective_cccs(
    STABLE_CCC_PATH, stability_threshold=PROTECTIVE_STABILITY
)
assert (protective_cccs.full_coefficient < 0).all()
assert (protective_cccs.sign_selection_probability >= PROTECTIVE_STABILITY).all()
display(protective_cccs[[
    'sender', 'receiver', 'ligand', 'receptor', 'pathway',
    'full_coefficient', 'sign_selection_probability'
]])
print('保护性 CCC 数量：', len(protective_cccs))


In [ ]:
# 13. 无监督发现：局部 exact CCC → 稳健非负标准化 → repeat-NMF → 连续 hotspot
niche_manifest = run_st_niche_discovery(
    STABLE_CCC_PATH, VISIUM_INPUT_DIR, RCTD_DIR, ST_SAMPLES, NICHE_OUTPUT,
    stability_threshold=PROTECTIVE_STABILITY, k_values=NMF_K_VALUES,
    nmf_restarts=NMF_RESTARTS, nmf_max_iter=NMF_MAX_ITER, selected_k=NMF_SELECTED_K,
    hotspot_quantile=HOTSPOT_QUANTILE, minimum_niche_spots=MIN_NICHE_SPOTS,
    distance_threshold=COMMOT_DISTANCE_THRESHOLD, cot_eps_p=COMMOT_COT_EPS_P,
    cot_rho=COMMOT_COT_RHO, cot_nitermax=COMMOT_COT_NITERMAX,
    random_seed=RANDOM_SEED, overwrite=NICHE_OVERWRITE,
)
assert niche_manifest['stage'] == 'label_free_discovery_complete'
assert niche_manifest['tls_labels_used'] is False
display(pd.Series(niche_manifest, name='value').to_frame())


In [ ]:
# 14. 查看 K 选择、program 组成、candidate niche 和空间图
k_diagnostics = pd.read_csv(NICHE_OUTPUT / 'nmf_k_diagnostics.tsv', sep='\t')
composition = pd.read_csv(NICHE_OUTPUT / 'program_composition.tsv', sep='\t')
candidate_niches = pd.read_csv(NICHE_OUTPUT / 'candidate_niches.tsv', sep='\t')
display(k_diagnostics)
display(composition.query('loading_rank <= 10')[[
    'program', 'loading_rank', 'loading', 'sender', 'receiver', 'ligand', 'receptor', 'pathway'
]])
display(candidate_niches.sort_values('mean_smoothed_activity', ascending=False).head(50))

fig = plot_nmf_diagnostics(NICHE_OUTPUT)
fig.savefig(NICHE_OUTPUT / 'nmf_k_diagnostics.png', dpi=180, bbox_inches='tight')
display(fig)
fig = plot_program_loadings(NICHE_OUTPUT, top_n=10)
fig.savefig(NICHE_OUTPUT / 'program_ccc_loadings.png', dpi=180, bbox_inches='tight')
display(fig)
fig = plot_spatial_programs(NICHE_OUTPUT, samples=ST_SAMPLES, image_root=ST_ROOT)
fig.savefig(NICHE_OUTPUT / 'program_spatial_hotspots.png', dpi=180, bbox_inches='tight')
display(fig)


In [ ]:
# 15. TLS 仅作 discovery 完成后的 held-out 验证
if TLS_ANNOTATION is None:
    print('TLS validation 暂未运行：请提供含 sample_id / spot_id / tls 的逐 spot 官方标注。')
    print('无标签 niche discovery 已完整保存于：', NICHE_OUTPUT)
else:
    tls_result = validate_tls_niches(
        NICHE_OUTPUT, TLS_ANNOTATION, permutations=TLS_PERMUTATIONS, random_seed=RANDOM_SEED
    )
    display(tls_result['program'].sort_values(['sample_id', 'roc_auc'], ascending=[True, False]))
    display(tls_result['niche'].sort_values(
        ['random_connected_p', 'tls_fraction_in_niche'], ascending=[True, False]
    ))
    fig = plot_tls_overlap(NICHE_OUTPUT, TLS_ANNOTATION, image_root=ST_ROOT)
    fig.savefig(NICHE_OUTPUT / 'tls_overlap_validation.png', dpi=180, bbox_inches='tight')
    display(fig)
    print('Program-level TLS validation:', NICHE_OUTPUT / 'tls_program_validation.tsv')
    print('Niche-level TLS validation:', NICHE_OUTPUT / 'tls_niche_validation.tsv')


## 16. GSE327192 单细胞 MERFISH functional niche（独立于前面的 Visium 流程）

Discovery 阶段只使用 11 条稳定保护性 CCC、单细胞表达/类型和空间邻接；TLS-like 标签只在 discovery manifest 冻结后重建并验证。

In [ ]:
# 16.1 MERFISH niche 参数（集中修改）
from IPython.display import Image, display
from cccphe.merfish_st_niche import run_discovery, run_posthoc_tls_validation

MERFISH_RAW_ROOT = Path('/home/nas3/biod/xueshuailin/New_Projects/data/GSE327192_HCC_MERFISH/raw')
MERFISH_LABEL_ROOT = Path('/home/nas3/biod/xueshuailin/CCC_Phe/data/processed/lihc_v0_merfish_anchor/merfish_anchor/cell_labels')
MERFISH_PROTECTIVE_CCC = Path('/home/nas3/biod/xueshuailin/CCC_Phe/results/lihc_v0_merfish_anchor/network_cox/stable_protective_ccc.tsv')
MERFISH_NICHE_OUTPUT = Path('/home/nas3/biod/xueshuailin/CCC_Phe/results/lihc_v0_merfish_anchor/merfish_functional_niche')
MERFISH_OVERWRITE_PROFILES = False  # 仅当原始表达/细胞标签变化时改为 True


In [ ]:
# 16.2 TLS-blind discovery：真实 sender cell → receiver cell、repeat-NMF、连续 hotspot
merfish_discovery = run_discovery(
    MERFISH_RAW_ROOT, MERFISH_LABEL_ROOT, MERFISH_PROTECTIVE_CCC,
    MERFISH_NICHE_OUTPUT, overwrite_profiles=MERFISH_OVERWRITE_PROFILES,
)
display(pd.Series(merfish_discovery, name='value').to_frame())
display(pd.read_csv(MERFISH_NICHE_OUTPUT / 'nmf_k_diagnostics.tsv', sep='\t'))
display(pd.read_csv(MERFISH_NICHE_OUTPUT / 'program_composition.tsv', sep='\t').query('rank <= 3'))
display(pd.read_csv(MERFISH_NICHE_OUTPUT / 'candidate_niches.tsv', sep='\t'))


In [ ]:
# 16.3 discovery 冻结后才运行：重建 TLS-like proxy 并作 held-out 验证
merfish_tls = run_posthoc_tls_validation(MERFISH_NICHE_OUTPUT)
display(pd.Series(merfish_tls, name='value').to_frame())
display(pd.read_csv(MERFISH_NICHE_OUTPUT / 'tls_program_validation.tsv', sep='\t'))
display(pd.read_csv(MERFISH_NICHE_OUTPUT / 'tls_niche_validation.tsv', sep='\t'))
display(Image(filename=str(MERFISH_NICHE_OUTPUT / 'program_ccc_loadings.png')))
display(Image(filename=str(MERFISH_NICHE_OUTPUT / 'candidate_niche_program_spatial.png')))
